- Linear / Regularized Linear Models
- Ridge
- Lasso
- LassoCV
- RidgeCV
- ElasticNetCV
- BayesianRidge
- HuberRegressor
- SGDRegressor
- PassiveAggressiveRegressor
- QuantileRegressor
- PoissonRegressor
- GammaRegressor
- TweedieRegressor
- OrthogonalMatchingPursuit


- Tree-Based Models
- DecisionTreeRegressor
- RandomForestRegressor
- ExtraTreesRegressor
- GradientBoostingRegressor
- HistGradientBoostingRegressor
- AdaBoostRegressor
- BaggingRegressor


- Advanced Boosting Libraries
- XGBRegressor from XGBoost
- LGBMRegressor from LightGBM
- CatBoostRegressor from CatBoost


- Kernel / Margin-Based Models
- SVR
- NuSVR
- KernelRidge


- KNeighborsRegressor
- RadiusNeighborsRegressor


- MLPRegressor



You could also wrap deep learning models manually if they expose fit() and predict().

- Probabilistic / Bayesian Models
- GaussianProcessRegressor
- BayesianRidge
- ARDRegression


- RANSACRegressor
- TheilSenRegressor
- HuberRegressor



- Ensemble Meta-Models
- VotingRegressor
- StackingRegressor

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import SVR

!pip install merf
!pip install sklearn.metrics
from sklearn.metrics import mean_squared_error
from scipy.stats import wilcoxon

from merf.merf import MERF

# -----------------------------
# Paths
# -----------------------------
dataset = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/radar_model_dataset_raw_features.csv")

RESULTS_PATH = Path(
    "C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics/SVM/MERF/RADAR"
)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(dataset)

# -----------------------------
# Clean data
# -----------------------------
df["recording_date"] = pd.to_datetime(df["recording_date"], errors="coerce")
df["participant_id"] = df["participant_id"].astype(str).str.strip()

# -----------------------------
# Outcome + grouping
# -----------------------------
y_score = df["phq8_score"]
ID_clusters = df["participant_id"]

# -----------------------------
# Random-effect covariates (Z)
# -----------------------------
z_candidates = [
    "Age",
    "Gender",
    "Education_Years",
    "Height"
]

z_features = [c for c in z_candidates if c in df.columns]
Z_factors = df[z_features].copy()

# -----------------------------
# Fixed-effect speech features (X)
# -----------------------------
x_candidates = [
    "Speaking_Rate",
    "Articulation_Rate",
    "Phonation_Ratio",
    "Pause_Rate",
    "Pause_Ratio",
    "mean_F0",
    "stdev_F0_Semitone",
    "HNR_dB",
    "Spectral_Slope",
    "Spectral_Tilt",
    "Cepstral_Peak_Prominence",
    "mean_F1_Loc",
    "std_F1_Loc",
    "mean_B1_Loc",
    "std_B1_Loc",
    "mean_F2_Loc",
    "std_F2_Loc",
    "mean_B2_Loc",
    "std_B2_Loc",
    "Spectral_Gravity",
    "Spectral_Std_Dev"
]

feature_cols = [c for c in x_candidates if c in df.columns]

# -----------------------------
# Remove missing values
# -----------------------------
required_cols = feature_cols + z_features + ["phq8_score", "participant_id"]

df_clean = df.dropna(subset=required_cols).copy()

X = df_clean[feature_cols]
Z_factors = df_clean[z_features]
y_score = df_clean["phq8_score"]
ID_clusters = df_clean["participant_id"]

print("Rows:", len(df_clean))
print("Participants:", df_clean["participant_id"].nunique())
print("Features used:", feature_cols)
print("Random-effect covariates:", z_features)

# -----------------------------
# GroupKFold CV
# -----------------------------
gkf = GroupKFold(n_splits=5)

rmse_list = []
fold_results = []

for fold, (train_index, test_index) in enumerate(
    gkf.split(X, y_score, groups=ID_clusters), start=1
):
    print(f"\nProcessing fold {fold}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y_score.iloc[train_index], y_score.iloc[test_index]

    clusters_train = ID_clusters.iloc[train_index]
    clusters_test = ID_clusters.iloc[test_index]

    Z_train = Z_factors.iloc[train_index]
    Z_test = Z_factors.iloc[test_index]

    # Scale X only
    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # MERF model
    rf_reg = SVR(
        n_estimators=500,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    )

    merf_model = MERF(
        fixed_effects_model=rf_reg,
        max_iterations=20
    )

    merf_model.fit(
        X_train_scaled,
        Z_train,
        clusters_train,
        y_train
    )

    y_pred = merf_model.predict(
        X_test_scaled,
        Z_test,
        clusters_test
    )

    mse = mean_squared_error(
        y_test,
        y_pred,
    )
    rmse = np.sqrt(mse)
    rmse_list.append(rmse)

    # Paired Wilcoxon: |y - ŷ_MERF| vs |y - ȳ_train| (per-sample; tests whether MERF fits better than mean baseline)
    baseline = np.full_like(y_test, fill_value=np.mean(y_train), dtype=float)
    e_merf = np.abs(np.asarray(y_test) - np.asarray(y_pred))
    e_mean = np.abs(np.asarray(y_test) - baseline)
    if len(e_merf) >= 2 and not np.allclose(e_merf, e_mean):
        w_res = wilcoxon(e_merf, e_mean, zero_method="wilcox", mode="auto")
        wilcoxon_p = float(w_res.pvalue)
    else:
        wilcoxon_p = float("nan")
    print(f"  Wilcoxon p (|y-ŷ_MERF| vs |y-ȳ_train|), fold {fold}:", wilcoxon_p)
    if wilcoxon_p < 0.05:
        print("    MERF abs errors differ from mean baseline (alpha=0.05).")
    else:
        print("    No significant difference from mean baseline at alpha=0.05 (improvement may be chance on this split).")

    fold_results.append({
        "fold": fold,
        "n_train": len(train_index),
        "n_test": len(test_index),
        "rmse": rmse,
        "wilcoxon_p_vs_train_mean_baseline": wilcoxon_p
    })

    print(f"Fold RMSE = {rmse:.2f}")

# -----------------------------
# Summary
# -----------------------------
fold_results_df = pd.DataFrame(fold_results)

summary_df = pd.DataFrame([{
    "subset": "all",
    "n_rows": len(df_clean),
    "n_participants": df_clean["participant_id"].nunique(),
    "rmse_mean": np.mean(rmse_list),
    "rmse_std": np.std(rmse_list)
}])

print("\nFold results:")
print(fold_results_df)

print("\nSummary:")
print(summary_df)

# -----------------------------
# Save results
# -----------------------------
fold_results_df.to_csv(
    RESULTS_PATH / "radar_merf_svr_cv_folds.csv",
    index=False
)

summary_df.to_csv(
    RESULTS_PATH / "radar_merf_svr_summary.csv",
    index=False
)

print("\nResults saved to:")
print(RESULTS_PATH)

ImportError: cannot import name 'SVR' from 'sklearn.ensemble' (c:\Users\janku\Documents\KCL\Research Project\Research Project\.venv\Lib\site-packages\sklearn\ensemble\__init__.py)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import SVR
from sklearn.metrics import mean_squared_error
from scipy.stats import wilcoxon

from merf.merf import MERF

# -----------------------------
# Paths
# -----------------------------
dataset = Path(
    "C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed/androids_model_dataset_basic.csv"
)

RESULTS_PATH = Path(
    "C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics/MERF/ANDROIDS"
)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(dataset)

# -----------------------------
# Clean columns
# -----------------------------
df["file_stem"] = df["file_stem"].astype(str).str.strip()
df["speech_type"] = df["speech_type"].astype(str).str.strip().str.lower()
df["subgroup_from_path"] = df["subgroup_from_path"].astype(str).str.strip().str.upper()
df["bdi_score"] = pd.to_numeric(df["bdi_score"], errors="coerce")

# -----------------------------
# Fixed-effect speech features (X)
# -----------------------------
feature_cols = [f"mfcc_{i}" for i in range(1, 14)] + ["pitch_mean", "energy_mean"]
feature_cols = [c for c in feature_cols if c in df.columns]

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# -----------------------------
# Random-effect covariates (Z)
# -----------------------------
z_candidates = ["speech_type", "subgroup_from_path"]
z_features = [c for c in z_candidates if c in df.columns]

# -----------------------------
# Remove missing values
# -----------------------------
required_cols = feature_cols + ["bdi_score", "file_stem"] + z_features
df_clean = df.dropna(subset=required_cols).copy()

# -----------------------------
# Build model inputs
# -----------------------------
X = df_clean[feature_cols].astype(float)
y_score = df_clean["bdi_score"].astype(float)
ID_clusters = df_clean["file_stem"]

Z_factors = pd.get_dummies(
    df_clean[z_features],
    drop_first=True
).astype(float)

print("Rows:", len(df_clean))
print("Unique groups:", df_clean["file_stem"].nunique())
print("Features used:", feature_cols)
print("Random-effect covariates:", z_features)

# -----------------------------
# GroupKFold CV
# -----------------------------
gkf = GroupKFold(n_splits=5)

rmse_list = []
fold_results = []

for fold, (train_index, test_index) in enumerate(
    gkf.split(X, y_score, groups=ID_clusters), start=1
):
    print(f"\nProcessing fold {fold}")

    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]

    y_train = y_score.iloc[train_index].astype(float)
    y_test = y_score.iloc[test_index].astype(float)

    clusters_train = ID_clusters.iloc[train_index]
    clusters_test = ID_clusters.iloc[test_index]

    Z_train = Z_factors.iloc[train_index].astype(float)
    Z_test = Z_factors.iloc[test_index].astype(float)

    # Scale X only
    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # MERF model
    rf_reg = SVR(
        n_estimators=500,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    )

    merf_model = MERF(
        fixed_effects_model=rf_reg,
        max_iterations=20
    )

    merf_model.fit(
        X_train_scaled,
        Z_train,
        clusters_train,
        y_train
    )

    y_pred = merf_model.predict(
        X_test_scaled,
        Z_test,
        clusters_test
    )

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    rmse_list.append(rmse)

    # Paired Wilcoxon: |y - ŷ_MERF| vs |y - ȳ_train|
    baseline = np.full_like(y_test, fill_value=np.mean(y_train), dtype=float)
    e_merf = np.abs(np.asarray(y_test) - np.asarray(y_pred))
    e_mean = np.abs(np.asarray(y_test) - baseline)
    if len(e_merf) >= 2 and not np.allclose(e_merf, e_mean):
        w_res = wilcoxon(e_merf, e_mean, zero_method="wilcox", mode="auto")
        wilcoxon_p = float(w_res.pvalue)
    else:
        wilcoxon_p = float("nan")
    print(f"  Wilcoxon p (|y-ŷ_MERF| vs |y-ȳ_train|), fold {fold}:", wilcoxon_p)
    if wilcoxon_p < 0.05:
        print("    MERF abs errors differ from mean baseline (alpha=0.05).")
    else:
        print("    No significant difference from mean baseline at alpha=0.05 (improvement may be chance on this split).")

    fold_results.append({
        "fold": fold,
        "n_train": len(train_index),
        "n_test": len(test_index),
        "rmse": rmse,
        "wilcoxon_p_vs_train_mean_baseline": wilcoxon_p
    })

    print(f"Fold RMSE = {rmse:.2f}")

# -----------------------------
# Summary
# -----------------------------
fold_results_df = pd.DataFrame(fold_results)

summary_df = pd.DataFrame([{
    "subset": "all",
    "n_rows": len(df_clean),
    "n_groups": df_clean["file_stem"].nunique(),
    "rmse_mean": np.mean(rmse_list),
    "rmse_std": np.std(rmse_list)
}])

print("\nFold results:")
print(fold_results_df)

print("\nSummary:")
print(summary_df)

# -----------------------------
# Save results
# -----------------------------
fold_results_df.to_csv(
    RESULTS_PATH / "androids_merf_cv_folds.csv",
    index=False
)

summary_df.to_csv(
    RESULTS_PATH / "androids_merf_summary.csv",
    index=False
)

print("\nResults saved to:")
print(RESULTS_PATH)